# DSAR × Lakeflow Declarative Pipelines · 02 · Erasure (all subjects, all layers)

Processes the **`dsar_request`** queue. For **every PENDING subject** it erases the
subject's records from **every layer** — `raw_user`, `bronze_user`, `silver_user`
(base tables) — then **refreshes** the `gold_user` materialized view, physically
purges, validates no trace, and marks the request COMPLETE.

This is the same erasure behavior as the base [`dsar_erasure/`](../../dsar_erasure)
layer, adapted so it runs **against a live streaming pipeline with zero downtime**:
because `01`/`01b` set `skipChangeCommits` on every streaming read, each DELETE is
skipped by the streams instead of crashing them.

### How subjects are matched

Intake gives an **email**. Bronze/silver have already redacted the email, so we
resolve **email → stable `user_id`** from `raw_user` once per subject, then erase by
`user_id` at every layer. (`user_id` is a non-PII key that survives masking.)

### Two request types (per row in `dsar_request`)

- **DELETE** — remove the subject's rows entirely.
- **OBFUSCATE** — keep rows, redact PII cells (only `raw_user` still holds cleartext
  PII; downstream is already masked).

### Gold is a materialized view

You cannot `DELETE` from a view. We erase the **base tables** and **refresh** the
gold MV so it re-derives clean from the erased silver.

## 0. Configuration

In [ ]:
dbutils.widgets.removeAll()

In [ ]:
dbutils.widgets.text("catalog", "dkushari_uc", "1 Catalog")
dbutils.widgets.text("schema", "allegiant_air_sdp_dsar", "2 Schema (match the pipeline target)")
dbutils.widgets.text("volume", "raw_user", "3 Landing volume (Auto Loader source)")
dbutils.widgets.text("redaction_token", "***REDACTED***", "4 Redaction token")
dbutils.widgets.dropdown("dry_run", "true", ["true", "false"], "5 Dry run (preview only)")
dbutils.widgets.dropdown("do_purge", "true", ["true", "false"], "6 Physically VACUUM tables after erase")
dbutils.widgets.dropdown("scrub_files", "true", ["true", "false"], "7 Scrub volume source files")
dbutils.widgets.text("pipeline_id", "", "8 Pipeline id (to refresh gold MV; optional)")

CATALOG = dbutils.widgets.get("catalog").strip()
SCHEMA  = dbutils.widgets.get("schema").strip()
VOLUME  = dbutils.widgets.get("volume").strip()
FQ      = f"{CATALOG}.{SCHEMA}"
TOKEN   = dbutils.widgets.get("redaction_token")
DRY     = dbutils.widgets.get("dry_run") == "true"
PURGE   = dbutils.widgets.get("do_purge") == "true"
SCRUB   = dbutils.widgets.get("scrub_files") == "true"
PIPELINE_ID = dbutils.widgets.get("pipeline_id").strip()

LANDING = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/landing"   # Auto Loader source files
SCRUB_STAGING = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/_scrub_staging"   # temp dir OUTSIDE landing/ (never ingested)
BASE_TABLES = ["silver_user", "bronze_user", "raw_user"]  # erase top-down; gold MV refreshed after
MV = "gold_user"
print("Schema:", FQ, "| landing:", LANDING, "| dry_run:", DRY, "| purge:", PURGE, "| scrub_files:", SCRUB)


## 1. Load the PENDING DSAR requests

Each row names one subject (email) and a request_type. We process them all in this
run — the multi-subject behavior you get in the base layer.

In [ ]:
from pyspark.sql import functions as F

def sqlstr(s):
    return "'" + str(s).replace("'", "''") + "'"

reqs = (spark.table(f"{FQ}.dsar_request")
        .where(F.col("status") == "PENDING")
        .select("request_id", "subject_email", "request_type")
        .collect())

if not reqs:
    print("No PENDING requests. Nothing to do.")
else:
    print(f"{len(reqs)} PENDING request(s):")
    for r in reqs:
        print(f"  {r['request_id']}  {r['subject_email']:<32} {r['request_type']}")


## 2. Resolve each subject email → user_id (from raw)

`raw_user` still holds cleartext email. We build, per request, the set of
`user_id`s to erase. (A subject with no match in raw was likely already erased.)

In [ ]:
plan = []   # list of dicts: {request_id, email, rtype, user_ids}
for r in reqs:
    email = r["subject_email"].strip().lower()
    ids = [x["user_id"] for x in
           spark.table(f"{FQ}.raw_user").where(F.lower("email") == email)
                .select("user_id").distinct().collect()]
    plan.append({"request_id": r["request_id"], "email": email,
                 "rtype": r["request_type"].strip().upper(), "user_ids": ids})
    print(f"  {r['request_id']}  {email:<32} {r['request_type']:<9} -> user_id(s): {ids or '(none)'}")

all_ids = sorted({u for p in plan for u in p["user_ids"]})
print("\nDistinct user_id(s) across all requests:", all_ids or "(none)")

## 3. Pre-count matches at every layer

So the erasure is auditable. Every table keys on `user_id`.

In [ ]:
if not all_ids:
    print("Nothing to erase.")
else:
    in_all = ", ".join(sqlstr(u) for u in all_ids)
    print("Rows matching any requested subject, per layer:")
    for t in BASE_TABLES + [MV]:
        try:
            n = spark.sql(f"SELECT count(*) c FROM {FQ}.{t} WHERE user_id IN ({in_all})").collect()[0]["c"]
            kind = "(MV, derived)" if t == MV else "(base table)"
            print(f"  {t:<14} {n}   {kind}")
        except Exception as e:
            print(f"  {t:<14} (missing) {str(e).splitlines()[0][:50]}")

## 4. Erase every subject from the base tables

Per request: **DELETE** removes the subject's rows; **OBFUSCATE** redacts PII cells
(only `raw_user` still has cleartext PII). Top-down silver → bronze → raw. Each
statement is a non-append commit the streams skip via `skipChangeCommits`. We erase
`raw_user` too, so a later full refresh can't resurrect the subject.

In [ ]:
def _mask_json(col):
    e = f"regexp_replace({col}, '(\"email\" *: *)\"[^\"]*\"', '$1\"{TOKEN}\"')"
    e = f"regexp_replace({e}, '(\"name\" *: *)\"[^\"]*\"', '$1\"{TOKEN}\"')"
    return e

modified = set()

def erase_stmt(table, user_ids, rtype):
    ul = ", ".join(sqlstr(u) for u in user_ids)
    pred = f"user_id IN ({ul})"
    if rtype == "DELETE":
        return f"DELETE FROM {FQ}.{table} WHERE {pred}"
    # OBFUSCATE: every layer carries cleartext PII in this demo, so redact the
    # subject's PII cells + in-JSON PII at EVERY base table (symmetric with DELETE).
    sets = [f"email = {sqlstr(TOKEN)}", f"full_name = {sqlstr(TOKEN)}",
            f"profile_json = {_mask_json('profile_json')}"]
    return f"UPDATE {FQ}.{table} SET {', '.join(sets)} WHERE {pred}"

for p in plan:
    if not p["user_ids"]:
        print(f"[skip] {p['request_id']} {p['email']}: no rows (already erased?)")
        continue
    print(f"\n=== {p['request_id']}  {p['email']}  ({p['rtype']}) ===")
    for t in BASE_TABLES:
        stmt = erase_stmt(t, p["user_ids"], p["rtype"])
        print(f"  [{'DRY' if DRY else 'RUN'}] {stmt}")
        if not DRY:
            spark.sql(stmt); modified.add(t)

print("\nBase tables modified:", sorted(modified) or "(none / dry run)")


## 5. Physical purge (CCPA "no trace")

Compact + VACUUM the base tables we modified. Serverless-safe technique (same as
base `03_physical_purge`): set table property
`delta.deletedFileRetentionDuration='interval 0 hours'` then a plain `VACUUM`
(`RETAIN 0 HOURS` trips the safety check on serverless).

In [ ]:
def purge(table):
    fqt = f"{FQ}.{table}"
    spark.sql(f"ALTER TABLE {fqt} SET TBLPROPERTIES ('delta.deletedFileRetentionDuration' = 'interval 0 hours')")
    try:
        spark.sql(f"REORG TABLE {fqt} APPLY (PURGE)")
    except Exception as e:
        print(f"  REORG skipped on {table}:", str(e).splitlines()[0][:70])
    spark.sql(f"VACUUM {fqt}")
    print(f"  purged {table}")

if DRY:
    print("Dry run — skipping purge.")
elif not PURGE:
    print("do_purge=false — skipping VACUUM.")
elif not modified:
    print("Nothing modified — skipping purge.")
else:
    for t in sorted(modified):
        try: purge(t)
        except Exception as e: print(f"  purge failed on {t}:", str(e).splitlines()[0][:70])

## 5b. Scrub the Auto Loader SOURCE FILES in the volume (CCPA — true erasure)

Erasing the *tables* is not enough. `raw_user` is ingested by **Auto Loader from JSON
files** in the volume landing zone — the cleartext PII still lives in those files, and
a **full refresh re-reads them**, resurrecting the subject. So for real CCPA erasure we
rewrite the landing files **in place**:

- We locate only the files that actually contain a requested subject (via Spark's
  `_metadata.file_path`), so we never touch other customers' files.
- **DELETE** → rewrite the file with the subject's records removed (drop the file if it
  becomes empty).
- **OBFUSCATE** → rewrite the file with the subject's PII cells + in-JSON PII redacted
  to the token (records kept).
- Auto Loader keys off file path + size/mtime, so a rewritten file is re-ingested on the
  next run — but since the tables already reflect the erasure and the rewritten file no
  longer holds cleartext, nothing is resurrected. Full refresh is now safe.


In [ ]:
import json as _json, os, uuid

def _volume_path(fp):
    """Normalize Spark's _metadata.file_path to a clean /Volumes/... FUSE path.
    file_path may carry a scheme (e.g. 'dbfs:/Volumes/...'); take the substring from
    '/Volumes/' onward so plain Python file I/O works on the volume mount."""
    i = fp.find("/Volumes/")
    return fp[i:] if i >= 0 else fp

def _redact_record(rec, rtype):
    """Return the record dict after erasure, or None to drop it (DELETE)."""
    if rtype == "DELETE":
        return None
    rec = dict(rec)                          # OBFUSCATE: redact PII, keep the record
    if "email" in rec:     rec["email"] = TOKEN
    if "full_name" in rec: rec["full_name"] = TOKEN
    pj = rec.get("profile_json")
    if isinstance(pj, str) and pj:
        try:
            obj = _json.loads(pj)
            c = obj.get("contact")
            if isinstance(c, dict):
                if "email" in c: c["email"] = TOKEN
                if "name"  in c: c["name"]  = TOKEN
            rec["profile_json"] = _json.dumps(obj, separators=(",", ":"))
        except Exception:
            pass  # fail-safe: leave as-is if unparseable (table-level scrub still applied)
    return rec

def _landing_reader():
    # only *.json (ignore markers like _SUCCESS and any stray temp), recurse subfolders
    return (spark.read.format("json")
            .option("recursiveFileLookup", "true")
            .option("pathGlobFilter", "*.json")
            .schema("event_id string, user_id string, email string, full_name string, "
                    "profile_json string, revenue double, event_ts string, _ingest_ts string"))

def _files_with_users(user_ids):
    """Distinct landing file paths (clean /Volumes/... ) that contain any of these user_ids."""
    ul = ", ".join(sqlstr(u) for u in user_ids)
    df = (_landing_reader().load(LANDING).where(f"user_id IN ({ul})")
          .select(F.col("_metadata.file_path").alias("fp")).distinct())
    return [_volume_path(r["fp"]) for r in df.collect()]

def _scrub_file(path):
    """Rewrite one landing file ATOMICALLY. Streams line-by-line to a temp file in a
    STAGING dir *outside* landing/ (so Auto Loader never ingests the temp), then
    os.replace()s it over the original. The original is untouched until the swap; on
    any error the temp is removed and the original left intact. Returns 'rewritten',
    or 'removed' if the file ends up empty (all records dropped)."""
    dbutils.fs.mkdirs(SCRUB_STAGING)
    tmp = f"{SCRUB_STAGING}/{uuid.uuid4().hex}.tmp"   # outside landing/, unique
    kept = 0
    try:
        with open(path, "r") as src, open(tmp, "w") as dst:
            for line in src:
                line = line.strip()
                if not line:
                    continue
                rec = _json.loads(line)
                uid = rec.get("user_id")
                if uid in uid_rtype:
                    rec = _redact_record(rec, uid_rtype[uid])
                    if rec is None:
                        continue   # DELETE -> drop record
                dst.write(_json.dumps(rec, separators=(",", ":")) + "\n")
                kept += 1
        if kept:
            os.replace(tmp, path)   # atomic swap (cross-dir replace verified on this volume)
            return "rewritten"
        os.remove(path)             # file emptied -> remove the original
        os.remove(tmp)              # and the (empty) temp
        return "removed"
    except Exception:
        if os.path.exists(tmp):     # never orphan a PII-bearing temp
            os.remove(tmp)
        raise                       # original untouched; surface loudly

# each user_id -> its request type (DELETE beats OBFUSCATE if ever both); computed once
uid_rtype = {}
for p in plan:
    for u in p["user_ids"]:
        if uid_rtype.get(u) != "DELETE":
            uid_rtype[u] = p["rtype"]

scrubbed_files, dropped_files = [], []
if not SCRUB:
    print("scrub_files=false — leaving volume source files untouched (NOT full-refresh-safe).")
elif DRY:
    if all_ids:
        files = _files_with_users(all_ids)
        print(f"[DRY] would scrub {len(files)} landing file(s) containing requested subjects:")
        for f in files: print("   ", f)
    else:
        print("[DRY] nothing to scrub.")
elif not all_ids:
    print("Nothing to scrub.")
else:
    target_files = _files_with_users(all_ids)
    print(f"Scrubbing {len(target_files)} landing file(s) (staging temp + atomic replace)...")
    for path in target_files:
        try:
            outcome = _scrub_file(path)
        except Exception as e:
            print(f"   ! FAILED scrubbing {path}: {str(e).splitlines()[0][:80]}")
            raise
        (scrubbed_files if outcome == "rewritten" else dropped_files).append(path)
    try: dbutils.fs.rm(SCRUB_STAGING, recurse=True)   # remove staging dir when done
    except Exception: pass
    print(f"  rewritten: {len(scrubbed_files)} file(s); removed (emptied): {len(dropped_files)} file(s)")
    for f in scrubbed_files: print("   ~", f)
    for f in dropped_files:  print("   x", f)


## 6. Refresh the gold materialized view

`gold_user` is a view derived from silver — we can't DELETE from it, and its stored
result still holds erased subjects until recomputed. Trigger a pipeline update so
it re-derives from the now-erased silver.

Set the `pipeline_id` widget to auto-refresh here; otherwise refresh from the
pipeline UI (Start), or `databricks pipelines start-update <id> --full-refresh`.

In [ ]:
def _discover_pipeline_id():
    """Auto-discover the Lakeflow pipeline that owns gold_user (so we can refresh it
    without the user pasting an id). Reads the 'pipelines.pipelineId' table property."""
    try:
        props = spark.sql(f"SHOW TBLPROPERTIES {FQ}.{MV}").collect()
        for r in props:
            if r["key"] == "pipelines.pipelineId":
                return r["value"]
    except Exception as e:
        print("  (could not read pipeline id from", MV, "->", str(e).splitlines()[0][:50], ")")
    return None

if DRY:
    print("Dry run — gold MV not refreshed.")
else:
    pid = PIPELINE_ID or _discover_pipeline_id()
    if not pid:
        print("No pipeline id (widget empty + none on gold_user). Refresh gold manually:")
        print("  UI: open the pipeline -> Start   (or Full refresh)")
        print("  CLI: databricks pipelines start-update <id> --full-refresh")
        print("Until refreshed, gold_user still shows erased subjects' aggregate rows.")
    else:
        from databricks.sdk import WorkspaceClient
        w = WorkspaceClient()
        upd = w.pipelines.start_update(pipeline_id=pid, full_refresh_selection=[MV])
        src = "widget" if PIPELINE_ID else f"auto-discovered from {MV}"
        print(f"Triggered refresh of {MV} (pipeline {pid}, {src}): {upd}.")
        print("Wait for the update to COMPLETE, then re-run section 7 to confirm gold is clean.")


## 7. Validate — no trace, and mark requests COMPLETE

Base-table counts must be **0** for DELETEd subjects; `gold_user` becomes 0 after
the MV refresh (section 6). We assert no cleartext email survives, then flip each
processed request to COMPLETE.

In [ ]:
if DRY:
    print("Dry run — no validation / status change. Re-run with dry_run=false.")
elif not reqs:
    print("Nothing was erased.")
else:
    del_ids = [u for pp in plan if pp["rtype"] == "DELETE" for u in pp["user_ids"]]
    if all_ids:
        in_all = ", ".join(sqlstr(u) for u in all_ids)
        print("Post-erasure counts (any requested subject):")
        for t in BASE_TABLES + [MV]:
            n = spark.sql(f"SELECT count(*) c FROM {FQ}.{t} WHERE user_id IN ({in_all})").collect()[0]["c"]
            # gold-stale note fires ONLY for DELETE subjects (OBFUSCATE keeps its rows by design)
            stale = False
            if t == MV and del_ids:
                din = ", ".join(sqlstr(u) for u in del_ids)
                stale = spark.sql(f"SELECT count(*) c FROM {FQ}.{t} WHERE user_id IN ({din})").collect()[0]["c"] > 0
            note = "  <- DELETE subject still in gold; refresh gold MV (section 6)" if stale else ""
            print(f"  {t:<14} {n}{note}")

    # --- DELETE subjects: no cleartext email + gone from every base table ---
    for p in plan:
        if p["rtype"] != "DELETE" or not p["user_ids"]:
            continue
        left = spark.sql(f"SELECT count(*) c FROM {FQ}.raw_user WHERE lower(email) = {sqlstr(p['email'])}").collect()[0]["c"]
        assert left == 0, f"cleartext {p['email']} survived in raw_user!"
        ul = ", ".join(sqlstr(u) for u in p["user_ids"])
        for t in BASE_TABLES:
            n = spark.sql(f"SELECT count(*) c FROM {FQ}.{t} WHERE user_id IN ({ul})").collect()[0]["c"]
            assert n == 0, f"{p['request_id']}: {t} still has {n} rows!"

    # --- OBFUSCATE subjects: rows kept at EVERY layer, but PII redacted (no cleartext trace) ---
    for p in plan:
        if p["rtype"] != "OBFUSCATE" or not p["user_ids"]:
            continue
        ul = ", ".join(sqlstr(u) for u in p["user_ids"])
        for t in BASE_TABLES:   # raw, bronze, silver all carry cleartext -> all must be redacted
            # original cleartext email must be gone from this layer
            bad_email = spark.sql(
                f"SELECT count(*) c FROM {FQ}.{t} WHERE lower(email) = {sqlstr(p['email'])}").collect()[0]["c"]
            assert bad_email == 0, f"{p['request_id']}: cleartext email {p['email']} survived OBFUSCATE in {t}!"
            # every kept row must have PII cells + in-JSON PII redacted to the token
            unmasked = spark.sql(
                f"SELECT count(*) c FROM {FQ}.{t} WHERE user_id IN ({ul}) AND "
                f"(email <> {sqlstr(TOKEN)} OR full_name <> {sqlstr(TOKEN)} "
                f"OR profile_json LIKE '%' || {sqlstr(p['email'])} || '%' "     # original email gone from JSON blob
                f"OR profile_json NOT LIKE '%' || {sqlstr(TOKEN)} || '%')"      # token present (redaction ran)
                ).collect()[0]["c"]
            assert unmasked == 0, f"{p['request_id']}: {unmasked} {t} row(s) still hold un-redacted PII after OBFUSCATE!"

    # --- VOLUME FILES: no cleartext trace of any requested subject (CCPA) ---
    if SCRUB and all_ids:
        import os
        landing_exists = True
        try:
            _ = dbutils.fs.ls(LANDING)
        except Exception:
            landing_exists = False
        if landing_exists:
            files_left = spark.read.format("json").option("recursiveFileLookup","true").option("pathGlobFilter","*.json").schema(
                "event_id string, user_id string, email string, full_name string, "
                "profile_json string, revenue double, event_ts string, _ingest_ts string"
            ).load(LANDING)
            # DELETE subjects: user_id must be gone from the files entirely (del_ids from above)
            if del_ids:
                ul = ", ".join(sqlstr(u) for u in del_ids)
                n = files_left.where(f"user_id IN ({ul})").count()
                assert n == 0, f"{n} landing-file record(s) for DELETE subjects survived the scrub!"
            # no original cleartext email of ANY requested subject remains in the files
            for pp in plan:
                if not pp["user_ids"]:
                    continue
                n = files_left.where(F.lower("email") == pp["email"]).count()
                assert n == 0, f"cleartext email {pp['email']} still present in landing files!"
                n2 = files_left.where(F.col("profile_json").contains(pp["email"])).count()
                assert n2 == 0, f"cleartext email {pp['email']} still present inside profile_json in landing files!"
            print("Volume files verified: no cleartext trace of any requested subject.")

    # --- mark ALL processed requests COMPLETE (incl. no-match: already satisfied) ---
    done_ids = [p["request_id"] for p in plan]          # every PENDING request handled this run
    nomatch  = [p["request_id"] for p in plan if not p["user_ids"]]
    if done_ids:
        idlist = ", ".join(sqlstr(i) for i in done_ids)
        spark.sql(f"UPDATE {FQ}.dsar_request SET status='COMPLETE' WHERE request_id IN ({idlist})")
    print("\nPASS — DELETE subjects erased with no trace; OBFUSCATE subjects redacted in place.")
    print("Requests marked COMPLETE:", done_ids)
    if nomatch:
        print("  (of those, no rows to erase — already satisfied:", nomatch, ")")
    display(spark.table(f"{FQ}.dsar_request"))


## 8. Idempotency

Because erasure removes subjects from the **base tables** (raw/bronze/silver), the
pipeline is idempotent under any mix of incremental and full-refresh updates: a
full refresh re-reads `raw_user` (subject already gone), so erased subjects never
reappear, and repeated updates converge to the same state. Re-running this notebook
is a no-op once the queue is COMPLETE.

In [ ]:
if not DRY and all_ids:
    in_all = ", ".join(sqlstr(u) for u in all_ids)
    del_ids = [u for pp in plan if pp["rtype"] == "DELETE" for u in pp["user_ids"]]
    print("Idempotency check — subject rows per layer:")
    for t in BASE_TABLES + [MV]:
        n = spark.sql(f"SELECT count(*) c FROM {FQ}.{t} WHERE user_id IN ({in_all})").collect()[0]["c"]
        stale = False
        if t == MV and del_ids:
            din = ", ".join(sqlstr(u) for u in del_ids)
            stale = spark.sql(f"SELECT count(*) c FROM {FQ}.{t} WHERE user_id IN ({din})").collect()[0]["c"] > 0
        flag = "  <- DELETE subject still in gold; run gold MV refresh (section 6)" if stale else ""
        print(f"  {t:<14} {n}{flag}")
else:
    print("Dry run — idempotency check skipped.")
